# Creating (nice) maps with GeoMakie
## Statement of need 
Creating maps for publications or presentations is an essential task in our everyday life as researchers.    
This set of notebooks describe how to generate professional maps using only Julia code.

## The packages
We will rely essentialy on the plotting library [Makie](https://makie.org/website/), and on [GeoMakie](https://geo.makie.org/stable/), to display information on a given map projection.    
Additionaly we will load [NCDatasets](https://github.com/JuliaGeo/NCDatasets.jl) to load some data.        

There are several backends. For instance:
- if you want to work OpenGL, then you need to load `GLMakie`
- if you want to render 2D by Cairo Graphics, you will use `CairoMakie`.

In [ ]:
using Makie
using CairoMakie
using GeoMakie
using NCDatasets
using JupyterFormatter
enable_autoformat()

## Data
We will load the GEBCO bathymetry.

In [ ]:
datafile = "gebco_30sec_16.nc"

if isfile(datafile)
    @info("File already downloaded")
else
    download("https://dox.uliege.be/index.php/s/U0pqyXhcQrXjEUX/download", datafile)
end

In order to slightly reduce the matrix size, we will sub-sample the dataset with a factor NN.

In [ ]:
NN = 4
ds = NCDataset(datafile)
lon = ds["lon"][1:NN:end]
lat = ds["lat"][1:NN:end]
bathy = ds["bat"][1:NN:end, 1:NN:end]
close(ds)
@info(size(bathy));

## Basic map with Makie only
A simple plot can be obtained either with
- `heatmap` or
- `surface`.

### Heatmap
In Makie, heatmap is the equivalent of `pcolor` in other languages.

In [ ]:
f = Figure()
ax = Axis(f[1, 1], title = "Bathymetry with Makie")
hm = heatmap!(ax, lon, lat, bathy)
contour!(ax, lon, lat, bathy, levels = [0.0], color = :black)
Colorbar(f[1, 2], hm, label = "(m)")
f

Let's detail all the operations:
- `f` is the figure object;
- `ax` is the Axis, which can contain many options;
- `heatmap!` is the function that generate the plot;
- `contour!` is added to plot the coastline (i.e. where the bathymetry is equal to zero.

#### What can be improved?
The result obtained with the default settings is always something we can improve.
1. The colormap: we will select one from [ColorSchemes](https://docs.juliaplots.org/dev/generated/colorschemes/)
2. The color range: the values are computed from the data, but we can set them _by hand_;
3. The figure size: we can increase the default value (600 x 450 CSS pixels in web contexts).
4. The colorbar label: it can be left unrotated (i.e., horizontal).

In [ ]:
using ColorSchemes
topocmap = ColorSchemes.delta

In [ ]:
f = Figure(size = (800, 500))
ax = Axis(f[1, 1], title = "Bathymetry with Makie")
hm = heatmap!(ax, lon, lat, bathy, colormap = topocmap, colorrange = [-5000, 5000])
contour!(ax, lon, lat, bathy, levels = [0.0], color = :black)
Colorbar(f[1, 2], hm, label = "(m)", labelrotation = 0)
f

### Surface
The function [`surface`](https://docs.makie.org/dev/reference/plots/surface) is another way to display 2D data. The default behaviour is different, as it performs interpolation and also shading effects. This can be avoided using the following options:    
- `shading=NoShading`
- `interpolate=false`

In [ ]:
f = Figure(size = (800, 500))
ax = Axis(f[1, 1], title = "Bathymetry with Makie")
hm = surface!(
    ax,
    lon,
    lat,
    bathy,
    colormap = topocmap,
    colorrange = [-5000, 5000],
    shading = NoShading,
    interpolate = false,
)
contour!(ax, lon, lat, bathy, levels = [0.0], color = :black)
Colorbar(f[1, 2], hm, label = "(m)", labelrotation = 0)
f

## Working with GeoMakie
The procedure for the creation of a plot is very similar, the main difference is that during the creation of the Axis (which is called here `GeoAxis`), one has to define a projection.
### Creating a projection
There are different ways to create a projection. They are defined as _proj-string_ (see https://proj.org/en/stable/operations/projections/index.html).
#### Global projections
We present a few examples using global projections

In [ ]:
globalproj1 = "+proj=ortho +lon_0=8.724647 +lat_0=42.583003"
globalproj2 = "+proj=merc"
globalproj3 = "+proj=moll"

In [ ]:
function make_map_proj(lon, lat, bathy, projection::String)
    f = Figure(size = (500, 500))
    ax = GeoAxis(f[1, 1], dest = projection, title = "Bathymetry with Makie")
    hm = surface!(
        ax,
        lon,
        lat,
        zeros(size(bathy));
        color = bathy,
        colormap = topocmap,
        colorrange = [-5000, 5000],
        shading = NoShading,
        interpolate = false,
    )
    contour!(ax, lon, lat, bathy, levels = [0.0], color = :black)
    Colorbar(f[1, 2], hm, label = "(m)", labelrotation = 0)
    display(f)
end

In [ ]:
make_map_proj(lon, lat, bathy, globalproj1)

In [ ]:
make_map_proj(globalproj2)

In [ ]:
make_map_proj(globalproj3)

## Local projections
There are plenty of choices for local projections, most of the time it is recommended to avoid Mercator

In [ ]:
domain = [0.88429, 4.85484, 38.38061, 40.486754050788335]

In [ ]:
ds = NCDataset(datafile)
lon = ds["lon"][:]
lat = ds["lat"][:]
goodlon = findall((lon .>= domain[1]) .& (lon .<= domain[2]))
goodlat = findall((lat .>= domain[3]) .& (lat .<= domain[4]))

lonregion = lon[goodlon]
latregion = lat[goodlat]
bathyregion = ds["bat"][goodlon, goodlat]
close(ds)
@info(size(bathycalvi));

In [ ]:
heatmap(lonregion, latregion, bathyregion)